# 02 — Guide walkthrough: three stars, from raw points to an Agreement, or none

The companion to `docs/guide/` (NEU-1800). Chapter 3 of the guide explains each piece of
evidence the pipeline produces; this notebook makes that evidence for three real stars from
the fixture set and writes every figure the guide shows to `docs/guide/figures/`, so each
one can be regenerated from the data it came from.

- **V0364 Dra**, an RR Lyrae star whose period every source agrees on: the clean case.
- **CSS_J181701.4+514940**, where VSX publishes 0.446574 d and the pipeline finds
  0.308586 d, a 1-day alias of it: the kind of correction Argelander exists to make.
- **CSS_J160212.7+530724**, a star whose convincing light curve belongs to the eclipsing
  binary 1.2″ away: a blend the pipeline refuses to vouch for.

Everything scientific is imported from `argelander`; the notebook only runs the pipeline,
reads its results out and draws them. It needs nothing but the committed fixtures
(`tests/fixtures/`), so it runs offline in the `jupyter` service. Each star takes most of a
minute, nearly all of it BoxLeastSquares over ZTF's seven-year baseline.

In [ ]:
import re
from pathlib import Path

import yaml
from astropy.table import Table
from IPython.display import Image, Markdown, display

from argelander.analysis.agreement import source_weight
from argelander.analysis.agreement import votes as cast_votes
from argelander.analysis.clean import sufficient
from argelander.analysis.pipeline import analyze
from argelander.catalog.gaia import records_from_tables
from argelander.lightcurve import load_target
from argelander.plots import render_all
from argelander.plots.combined import combined_figure
from argelander.plots.phase import phase_figure
from argelander.prototype import relation

REPO = Path("..").resolve()
FIXTURES = REPO / "tests" / "fixtures"
LIGHT_CURVES = FIXTURES / "lightcurves"
FIGURES = REPO / "docs" / "guide" / "figures"
DPI = 100

STARS = {s["star"]: s for s in yaml.safe_load((FIXTURES / "manifest.yaml").read_text())}
RECORDS = records_from_tables(
    *(
        Table.read(FIXTURES / "catalog" / f"{name}.xml", format="votable")
        for name in ("catalog", "cross_ids", "neighbours", "tess_field")
    )
)

## Helpers

`run` loads a star's fixtures and runs the pipeline (`analysis/pipeline.py`: clean →
periodograms → aliases → blend → Agreement → CMD), returning the raw Light Curves, the
Analysis and the cleaned Light Curves the plots draw. The rest turns pieces of the Analysis
into tables, or saves a figure under `docs/guide/figures/<star>/` and shows it here.

In [ ]:
def run(name: str):
    star = STARS[name]
    source_id = star["gaia_source_id"]
    raw = load_target(LIGHT_CURVES, source_id)
    analysis, cleaned = analyze(raw, RECORDS.get(source_id), target_id=source_id)
    return star, raw, analysis, cleaned


def figure_dir(name: str) -> Path:
    return FIGURES / re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")


def show(path: Path) -> None:
    display(Image(filename=path))


def save(fig, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=DPI)
    show(path)


def table(header: list[str], rows: list[list]) -> None:
    sep = ["---"] * len(header)
    lines = ["| " + " | ".join(header) + " |", "| " + " | ".join(sep) + " |"]
    lines += ["| " + " | ".join(str(c) for c in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))


def fmt(value, spec=".6f") -> str:
    return "–" if value is None else format(value, spec)


def points_table(raw, cleaned) -> None:
    cleaned_by_band = {(lc.survey, lc.band): lc for lc in cleaned}
    rows = []
    for lc in raw:
        after = cleaned_by_band[(lc.survey, lc.band)]
        rows.append(
            [
                f"{lc.survey} {lc.band}",
                lc.time.size,
                int((lc.quality != 0).sum()),
                after.time.size,
                f"{after.time.max() - after.time.min():.0f}",
                "yes" if sufficient(after) else "no (under 30 points)",
            ]
        )
    header = ["Band", "Points in the fixture", "Survey-flagged", "After cleaning", "Baseline (d)"]
    table(header + ["Searched"], rows)


def votes_table(analysis, record) -> None:
    contaminated = analysis.blend.surveys if analysis.blend else frozenset()
    gaia_period = record.vari_period if record else None
    rows = [
        [v.source, fmt(v.period), source_weight(v.source, contaminated)]
        for v in cast_votes(analysis.alias_tests, gaia_period)
    ]
    table(["Source", "Votes for (d)", "Weight"], rows)
    a = analysis.agreement
    if a is None:
        display(Markdown("**Agreement:** none. No period carries two sources' weight."))
    else:
        display(
            Markdown(
                f"**Agreement:** {a.agreed_period:.7f} d from {', '.join(a.agreed_sources)}, "
                f"relation {a.relation}, tolerance {a.tolerance:.2%}."
            )
        )


def ranking_table(analysis, survey: str, band: str, n: int = 8) -> None:
    ranking = next(t for t in analysis.alias_tests if (t.survey, t.band) == (survey, band)).ranking
    rows = [
        [i + 1, fmt(h.period), fmt(h.candidate), h.relation, h.bins]
        + [f"{h.scatter:.3f}", f"{h.score:.1f}"]
        for i, h in enumerate(ranking[:n])
    ]
    display(Markdown(f"**{survey} {band}**: the {n} best of {len(ranking)} hypotheses tried."))
    table(["Rank", "Period (d)", "From candidate", "Relation", "Bins", "Scatter", "Score"], rows)


def peaks_table(analysis, survey: str, band: str) -> None:
    rows = []
    for search in analysis.periods:
        if (search.survey, search.band) != (survey, band) or search.insufficient:
            continue
        for peak in search.peaks[:3]:
            fap = fmt(peak.false_alarm_probability, ".2g")
            rows.append([search.method, fmt(peak.period), f"{peak.power:.3g}", fap])
    table(["Method", "Peak period (d)", "Power", "False-alarm probability"], rows)


def contamination_table(analysis) -> None:
    if analysis.blend is None or not analysis.blend.contaminations:
        display(Markdown("No survey is flagged."))
        return
    rows = [
        [c.survey, c.flag, fmt(c.flux_fraction, ".2f"), c.remark]
        for c in analysis.blend.contaminations
    ]
    table(["Survey", "Flag", "Share of light not the Target's", "Remark for VSX"], rows)


def neighbours_table(record) -> None:
    rows = [[n.source_id, f"{n.separation:.1f}", fmt(n.g, ".2f")] for n in record.neighbours]
    header = ["Gaia DR3 neighbour", "Separation (″)", "G (mag)"]
    table(header, rows or [["none within 20″", "", ""]])
    f = record.tess_field
    display(
        Markdown(
            f"TESS field (120″): {f.n_sources} other sources, "
            f"brightest G = {fmt(f.brightest_g, '.2f')}, "
            f"all of them together G = {fmt(f.combined_g, '.2f')}; "
            f"the Target is G = {record.g:.2f}."
        )
    )


def fold_figure(cleaned, survey: str, band: str, period: float, vsx_type: str):
    searched = [lc for lc in cleaned if sufficient(lc)]
    epoch = min(float(lc.time[0]) for lc in searched)
    baseline = max(float(lc.time[-1]) for lc in searched) - epoch
    lc = next(lc for lc in cleaned if (lc.survey, lc.band) == (survey, band))
    return phase_figure(lc, period=period, epoch=epoch, vsx_type=vsx_type, baseline=baseline)

## V0364 Dra: the clean case

An RR Lyrae star (type `RRAB`) in Draco at G = 14.85. VSX publishes a period of 0.477038 d,
and Chen et al. (2020) found 0.4770406 d independently in ZTF, so the published value is
one the gate can trust. Three surveys observed it: ZTF in g and r, TESS, and Gaia in G, BP
and RP. ASAS-SN is a coverage gap until NEU-1796.

### Raw points

What the fetchers wrote to the Cache (trimmed to 500 points a band for the fixtures),
drawn on one time axis. TESS measures flux, not magnitude, so it is drawn as a magnitude
change from its own median and lifted to the level of the magnitude surveys.

In [ ]:
clean_star, v_raw, v_analysis, v_cleaned = run("V0364 Dra")
save(combined_figure(v_raw), figure_dir("V0364 Dra") / "raw.png")

### Cleaning

Survey quality flags first, then a 5σ clip around a running median, which in practice only
touches TESS's fast cadence. A band needs 30 surviving points to be searched; Gaia's few
dozen transits make it over the line here, but Gaia epoch photometry never votes on its
own (its published period does).

In [ ]:
points_table(v_raw, v_cleaned)

### The review-app plots

`render_all` writes exactly the files the review app will show for a Target: the
periodogram, one phase plot per searched band, the colour-magnitude diagram, the combined
light curve and `submission.png`, the composite the VSX form takes.

In [ ]:
v_plots = render_all(
    v_analysis,
    v_cleaned,
    vsx_type=clean_star["published_type"],
    period=v_analysis.agreement.agreed_period,
    epoch=None,
    out_dir=figure_dir("V0364 Dra"),
)
sorted(v_plots)

### The periodogram

One panel per band and method. The solid red line is the agreed period, the dashed orange
lines are the aliases the alias test tried, and the dotted blue line is Gaia's published
period, scored in every band. Lomb-Scargle finds a sinusoid; BLS looks for a box-shaped
dip, and is here for eclipsing binaries, not pulsators. In the ZTF panels there is a peak
under every dashed line: the same pulsation seen through once-a-night sampling.

In [ ]:
show(v_plots["periodogram"])
peaks_table(v_analysis, "ztf", "g")

### Aliases and folding

Every peak, and Gaia's period, is folded at itself and at its aliases: 2P, P/2 and the
1-day, half-day and sidereal-day aliases. The fold with the least scatter inside its phase
bins wins, with a penalty for using more bins (the score) so a longer period has to earn
them. Here the first-ranked hypothesis is the period itself.

In [ ]:
ranking_table(v_analysis, "ztf", "g")
show(v_plots["phase-ztf-g"])
show(v_plots["phase-tess-T"])

### Agreement

Each source votes with its best fold. Two independent sources within the tolerance of each
other settle the period; a contaminated survey counts half. Four sources land on 0.47704 d.

In [ ]:
votes_table(v_analysis, RECORDS[clean_star["gaia_source_id"]])

### Blend checks

Even the clean case carries a flag. `tess_contaminated` is real: TESS's 120″ aperture holds
other stars whose light adds up. ZTF is not flagged: its separate fields' zero points,
compared at equal phase once the period is settled, agree to 0.01 mag (NEU-1798). The
median rule it replaced read 0.12 mag here, from fields that sample the cycle unevenly.
No neighbour within 3″ is involved, as the neighbour table shows.

In [ ]:
contamination_table(v_analysis)
neighbours_table(RECORDS[clean_star["gaia_source_id"]])

### The colour-magnitude diagram

Gaia's parallax gives the distance, the distance turns apparent G into absolute G, and
BP−RP is the colour. The star lands on the instability strip, where RR Lyrae stars live,
which is the kind of independent support a period alone cannot give.

In [ ]:
show(v_plots["cmd"])
v_analysis.cmd

### What the VSX form gets

`submission.png`: the phase plot of the band with the most points above the combined
light curve, under the form's 1 MB limit.

In [ ]:
show(v_plots["submission"])

## CSS_J181701.4+514940: the alias VSX published

VSX gives this star 0.446574 d, type `EW`, from the Catalina survey's bulk import. Gaia's
eclipsing-binary pipeline gives 0.308582 d. One cycle per day separates them: ground
surveys observe once a night, so a periodogram cannot tell a signal at *f* from one at
*f* ± 1 cycle/day, and the bulk import took the wrong peak.

In [ ]:
alias_star, c181_raw, c181_analysis, c181_cleaned = run("CSS_J181701.4+514940")
vsx_period, true_period = alias_star["published_period"], alias_star["true_period"]
display(
    Markdown(
        f"VSX: {vsx_period} d → {1 / vsx_period:.4f} cycles/day. "
        f"Gaia: {true_period} d → {1 / true_period:.4f} cycles/day. "
        f"Difference: {abs(1 / vsx_period - 1 / true_period):.4f} cycles/day."
    )
)
c181_plots = render_all(
    c181_analysis,
    c181_cleaned,
    vsx_type=alias_star["published_type"],
    period=c181_analysis.agreement.agreed_period,
    epoch=None,
    out_dir=figure_dir("CSS_J181701.4+514940"),
)
show(c181_plots["periodogram"])

### Three folds of the same ZTF g points

At VSX's 0.446574 d the points spread over the full range at every phase and the median
wanders: the fold does not hold. At 0.154292 d, the Lomb-Scargle peak, the curve is clean
but single-humped: a contact binary's two eclipses have folded onto each other. At
0.30858 d, twice the peak, the two eclipses separate, and two dips per cycle where a
pulsator has one show it is an eclipsing binary.

In [ ]:
save(
    fold_figure(c181_cleaned, "ztf", "g", vsx_period, "EW"),
    figure_dir("CSS_J181701.4+514940") / "fold-at-vsx-period.png",
)
half = c181_analysis.agreement.agreed_period / 2
save(
    fold_figure(c181_cleaned, "ztf", "g", half, "EW"),
    figure_dir("CSS_J181701.4+514940") / "fold-at-half-period.png",
)
show(c181_plots["phase-ztf-g"])

### How the alias test ranks them

The ranking tries VSX's period too, as the `1d-` alias of the true one, and scores it far
below the true period and its half.

In [ ]:
ranking_table(c181_analysis, "ztf", "g", n=10)
ranking = next(t for t in c181_analysis.alias_tests if (t.survey, t.band) == ("ztf", "g")).ranking
near_vsx = [(i + 1, h) for i, h in enumerate(ranking) if relation(h.period, vsx_period) == "1:1"]
for rank, h in near_vsx:
    print(
        f"VSX's period is rank {rank} of {len(ranking)}: {h.period:.6f} d, "
        f"the {h.relation} alias of {h.candidate:.6f} d, scatter {h.scatter:.3f}"
    )

### Agreement at 2:1

ZTF g, ZTF r and TESS each vote for 0.154292 d, the peak; Gaia's published period votes
for 0.308582 d. Agreement sees the 2:1 relation, takes the longer period, which separates
the two eclipses, and records the relation for the classifier.

In [ ]:
votes_table(c181_analysis, RECORDS[alias_star["gaia_source_id"]])
contamination_table(c181_analysis)
show(c181_plots["cmd"])

## CSS_J160212.7+530724: a blend the pipeline refuses to trust

The VSX entry sits 0.6″ from a Gaia DR3 source that Gaia does not flag as variable. The
Gaia source 1.2″ from it is an eclipsing binary, two magnitudes brighter, whose Gaia period
matches the VSX one. Ground-based surveys cannot separate two stars that close: the
variability VSX credits to this star is the neighbour's.

In [ ]:
blend_star, c160_raw, c160_analysis, c160_cleaned = run("CSS_J160212.7+530724")
neighbours_table(RECORDS[blend_star["gaia_source_id"]])
c160_plots = render_all(
    c160_analysis,
    c160_cleaned,
    vsx_type=blend_star["published_type"],
    period=None,
    epoch=None,
    out_dir=figure_dir("CSS_J160212.7+530724"),
)
show(c160_plots["combined"])

### The signal is real; the star is wrong

Folded at VSX's 0.318016 d the ZTF g points make a convincing contact-binary curve. The
one hint in the light curve itself is the magnitude axis: Gaia puts this star at G = 15.7
and ZTF measures it near 14, because the aperture is mostly the neighbour's light. The
rest is the catalogue of positions.

In [ ]:
save(
    fold_figure(c160_cleaned, "ztf", "g", blend_star["published_period"], "EW"),
    figure_dir("CSS_J160212.7+530724") / "fold-at-vsx-period.png",
)
show(c160_plots["periodogram"])

### Every source at half weight

ZTF, TESS and ASAS-SN are all flagged, each by its own aperture; the ASAS-SN flag comes
from the catalogue alone, since no ASAS-SN Light Curve exists yet. Three votes at the same
period would normally be an easy Agreement; at half weight each they add up to 1.5, under
the two clean sources required, so there is no Agreement, no phase plot and no period to
propose. The remarks are the sentences VSX asks for when a range is contaminated.

In [ ]:
votes_table(c160_analysis, RECORDS[blend_star["gaia_source_id"]])
contamination_table(c160_analysis)

### Nothing to place on the diagram

Gaia has no BP−RP colour for this source, so the CMD region is unknown, and no Gaia epoch
photometry survived the fetch. `submission.png` falls back to the combined light curve
alone, as it does for any Target without an agreed period.

In [ ]:
show(c160_plots["cmd"])
c160_analysis.cmd

## Where the figures went

Every file under `docs/guide/figures/` that the guide shows, with the star it belongs to.
Re-running this notebook overwrites them.

In [ ]:
for path in sorted(FIGURES.rglob("*.png")):
    print(path.relative_to(REPO))